# LeetCode 3139 — Minimum Cost to Equalize Array bằng Python

- Link: https://leetcode.com/problems/minimum-cost-to-equalize-array/
- Độ khó: **Hard**
- Chủ đề: **Tham lam + duyệt giá trị đích** — với đích `T`, chỉ cần tổng thiếu hụt `S` và thiếu hụt lớn nhất `M`; số thao tác đôi tối đa là `min(⌊S/2⌋, S − M)`; thử mọi `T ∈ [max, 2·max]` (hoặc vài điểm tới hạn)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minCostToEqualizeArray(self, nums: List[int], cost1: int, cost2: int) -> int:
```

## 1. Mục tiêu học

1. Chỉ được **tăng** ⇒ giá trị chung cuối cùng `T ≥ max(nums)`; thiếu hụt `d_i = T − nums[i]`.
2. Nếu `2·cost1 ≤ cost2`, thao tác đôi không bao giờ có lợi ⇒ đáp án `S · cost1` với `T = max`.
3. Ngược lại, muốn dùng **nhiều thao tác đôi nhất**: mỗi thao tác đôi giảm hai thiếu hụt **khác chỉ số**; số cặp tối đa là `min(⌊S/2⌋, S − M)` (phần tử thiếu nhiều nhất chỉ ghép được với phần còn lại).
4. Tăng `T` làm `S` tăng `n` nhưng `M` chỉ tăng 1 ⇒ có thể **cân bằng** để ghép hết — đôi khi rẻ hơn dù tổng thiếu lớn hơn.
5. Duyệt `T` từ `max` tới `2·max` (`O(max)`), hoặc chỉ vài điểm tới hạn (`O(n)`); kiểm chứng bằng tìm kiếm vét cạn trên trạng thái nhỏ.

## 2. Đề bài & ví dụ

Cho `nums`, `cost1`, `cost2`. Thao tác: tăng một phần tử thêm 1 (giá `cost1`), hoặc tăng hai phần tử **khác chỉ số** thêm 1 mỗi cái (giá `cost2`). Tìm chi phí nhỏ nhất để mọi phần tử bằng nhau, mod `10^9 + 7`.

**Ví dụ 1:** `[4,1]`, `cost1 = 5`, `cost2 = 2` → `15`. **Ví dụ 2:** `[2,3,3,3,5]`, `2`, `1` → `6`. **Ví dụ 3:** `[3,5,3]`, `1`, `3` → `4`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`; `1 ≤ nums[i], cost1, cost2 ≤ 10^6`.

In [1]:
from typing import List
from functools import lru_cache
import random, time

MOD = 10 ** 9 + 7
examples = [([4, 1], 5, 2, 15), ([2, 3, 3, 3, 5], 2, 1, 6), ([3, 5, 3], 1, 3, 4)]
for a, c1, c2, exp in examples:
    print(a, c1, c2, '-> kỳ vọng', exp)

[4, 1] 5 2 -> kỳ vọng 15
[2, 3, 3, 3, 5] 2 1 -> kỳ vọng 6
[3, 5, 3] 1 3 -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Số cặp tối đa ghép được từ các thiếu hụt

Cho các thiếu hụt `d_1, …, d_n` với tổng `S`, lớn nhất `M`. Số thao tác đôi tối đa (mỗi thao tác lấy 1 từ hai chỉ số khác nhau) là `min(⌊S/2⌋, S − M)`:

- Nếu `M ≤ S − M`: luôn ghép được `⌊S/2⌋` cặp (lặp: ghép hai thiếu hụt lớn nhất).
- Nếu `M > S − M`: phần tử lớn nhất chỉ ghép với tổng `S − M` của phần còn lại ⇒ `S − M` cặp, dư `2M − S` đơn vị của phần tử lớn nhất.

In [2]:
def max_pairs_greedy(d):
    d = sorted(d, reverse=True); pairs = 0
    while len(d) >= 2 and d[1] > 0:
        d[0] -= 1; d[1] -= 1; pairs += 1
        d.sort(reverse=True)
    return pairs

rng = random.Random(3139)
for _ in range(300):
    d = [rng.randint(0, 6) for _ in range(rng.randint(1, 5))]
    S, M = sum(d), max(d)
    assert max_pairs_greedy(d) == min(S // 2, S - M)
print('Số cặp tối đa = min(⌊S/2⌋, S − M): khớp tham lam "ghép hai lớn nhất" trên 300 bộ.')

Số cặp tối đa = min(⌊S/2⌋, S − M): khớp tham lam "ghép hai lớn nhất" trên 300 bộ.


### 3.2. Chi phí với đích `T` cố định

Với `2·cost1 > cost2`, dùng tối đa số cặp `p` rồi phần còn lại đơn: `cost(T) = p · cost2 + (S − 2p) · cost1`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ: `[1, 10, 10]`, `cost1 = 10`, `cost2 = 1`. Với `T = 10`: thiếu `[9, 0, 0]` — không ghép được, tốn `90`. Với `T = 19`: thiếu `[18, 9, 9]`, `S = 36`, `M = 18 = S − M` ⇒ 18 cặp, tốn `18`! Nâng đích làm các phần tử "đã đủ" có thêm việc để **ghép cặp** với phần tử thiếu nhiều.

Mỗi khi `T` tăng 1: `S += n`, `M += 1` ⇒ `S − M` tăng `n − 1` — sớm muộn `M ≤ S − M`. Vượt qua điểm đó thì chi phí chỉ tăng (thêm `n` đơn vị cần lấp). Nên chỉ cần xét `T` tới khoảng `2·max`.

## 5. Cách 1 — Brute Force: tìm kiếm có nhớ trên vector thiếu hụt

In [3]:
def brute(nums, c1, c2, extra=6):
    mx = max(nums); best = None
    for T in range(mx, mx + extra + 1):
        @lru_cache(maxsize=None)
        def f(d):
            if not any(d):
                return 0
            L = list(d); res = float('inf')
            for i in range(len(L)):
                if L[i]:
                    a = L[:]; a[i] -= 1; res = min(res, c1 + f(tuple(sorted(a))))
                    for j in range(i + 1, len(L)):
                        if L[j]:
                            b = L[:]; b[i] -= 1; b[j] -= 1; res = min(res, c2 + f(tuple(sorted(b))))
            return res
        v = f(tuple(sorted(T - x for x in nums)))
        best = v if best is None else min(best, v)
    return best % MOD

for a, c1, c2, exp in examples:
    print('brute:', brute(a, c1, c2), '| kỳ vọng', exp)

brute: 15 | kỳ vọng 15
brute: 6 | kỳ vọng 6
brute: 4 | kỳ vọng 4


Brute không dùng công thức nào — thử **mọi** chuỗi thao tác (có nhớ) cho vài giá trị đích. Chỉ chạy được với giá trị nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

```
n ≤ 2: chỉ có thể tăng phần nhỏ bằng thao tác đơn ⇒ (max − min) · cost1
2·cost1 ≤ cost2: (n·max − sum) · cost1
ngược lại: với T ∈ [max, 2·max]:
    S = n·T − sum; M = T − min
    p = S // 2 nếu M ≤ S − M, ngược lại S − M
    cost = p·cost2 + (S − 2p)·cost1
lấy min, rồi mod
```

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Duyệt `T ∈ [max, 2·max]` với công thức (lời giải chính)

In [4]:
def cost_T(nums_sum, n, mn, c1, c2, T):
    S = n * T - nums_sum; M = T - mn
    if 2 * c1 <= c2:
        return S * c1
    p = S // 2 if M <= S - M else S - M
    return p * c2 + (S - 2 * p) * c1

def enumerate_T(nums, c1, c2, upto=2):
    n = len(nums); mx = max(nums); mn = min(nums); s = sum(nums)
    if n <= 2 or 2 * c1 <= c2:
        return (n * mx - s) * c1 % MOD
    return min(cost_T(s, n, mn, c1, c2, T) for T in range(mx, upto * mx + 1)) % MOD

for a, c1, c2, exp in examples:
    print('duyệt T:', enumerate_T(a, c1, c2), '| kỳ vọng', exp)

duyệt T: 15 | kỳ vọng 15
duyệt T: 6 | kỳ vọng 6
duyệt T: 4 | kỳ vọng 4


(`n ≤ 2`: với hai phần tử, nâng cả hai bằng thao tác đôi không làm giảm chênh lệch ⇒ chỉ thao tác đơn trên phần tử nhỏ; công thức chung cũng cho kết quả này với `T = max`.)

### 7.2. Cách 3 — Chỉ xét vài điểm tới hạn `O(n)`

Hai chế độ: khi `M > S − M` (chưa cân bằng), chi phí là hàm **tuyến tính** theo `T` ⇒ cực tiểu ở biên (`T = max` hoặc ngay trước điểm cân bằng). Khi đã cân bằng, chi phí `⌊S/2⌋·cost2 + (S mod 2)·cost1` tăng theo `T` trừ dao động chẵn lẻ ⇒ xét điểm cân bằng `T₀` và hai điểm kế. `T₀` là `T` nhỏ nhất `≥ max` với `2(T − min) ≤ nT − sum` ⇔ `T ≥ ⌈(sum − 2·min) / (n − 2)⌉`.

In [5]:
def critical_points(nums, c1, c2):
    n = len(nums); mx = max(nums); mn = min(nums); s = sum(nums)
    if n <= 2 or 2 * c1 <= c2:
        return (n * mx - s) * c1 % MOD
    T0 = max(mx, -(-(s - 2 * mn) // (n - 2)))
    cands = {mx, T0 - 1, T0, T0 + 1, T0 + 2}
    return min(cost_T(s, n, mn, c1, c2, T) for T in cands if T >= mx) % MOD

for a, c1, c2, exp in examples:
    print('điểm tới hạn:', critical_points(a, c1, c2), '| kỳ vọng', exp)

điểm tới hạn: 15 | kỳ vọng 15
điểm tới hạn: 6 | kỳ vọng 6
điểm tới hạn: 4 | kỳ vọng 4


### 7.3. Cách 4 — Duyệt `T` với tổng tham lam thật (không dùng công thức số cặp)

Cho dữ liệu vừa: với mỗi `T`, đếm số cặp bằng tham lam "ghép hai thiếu hụt lớn nhất" dùng heap — xác nhận công thức `min(⌊S/2⌋, S − M)` ở quy mô lớn hơn brute.

In [6]:
import heapq
def greedy_heap(nums, c1, c2, extra=None):
    n = len(nums); mx = max(nums)
    if n <= 2 or 2 * c1 <= c2:
        return (n * mx - sum(nums)) * c1 % MOD
    best = None
    for T in range(mx, 2 * mx + 1 if extra is None else mx + extra + 1):
        h = [-(T - x) for x in nums if T - x > 0]; heapq.heapify(h); pairs = 0; S = sum(T - x for x in nums)
        while len(h) >= 2:
            a = -heapq.heappop(h); b = -heapq.heappop(h)
            pairs += 1; a -= 1; b -= 1                 # ghép một cặp từ hai thiếu hụt lớn nhất
            if a: heapq.heappush(h, -a)
            if b: heapq.heappush(h, -b)
        v = pairs * c2 + (S - 2 * pairs) * c1
        best = v if best is None else min(best, v)
    return best % MOD

for a, c1, c2, exp in examples:
    print('heap tham lam:', greedy_heap(a, c1, c2), '| kỳ vọng', exp)

heap tham lam: 15 | kỳ vọng 15
heap tham lam: 6 | kỳ vọng 6
heap tham lam: 4 | kỳ vọng 4


## 8. TRACE từng bước

In chi phí theo `T` cho `[1, 10, 10]`, `cost1 = 10`, `cost2 = 1` để thấy điểm cân bằng.

In [7]:
a, c1, c2 = [1, 10, 10], 10, 1
n, s, mn = len(a), sum(a), min(a)
for T in range(10, 23):
    S = n * T - s; M = T - mn; p = S // 2 if M <= S - M else S - M
    print(f'T = {T:2d}: S = {S:2d}, M = {M:2d}, S − M = {S - M:2d} -> cặp {p:2d}, đơn {S - 2 * p:2d}, chi phí {cost_T(s, n, mn, c1, c2, T)}')
print('đáp án:', Solution().minCostToEqualizeArray(a, c1, c2) if 'Solution' in globals() else enumerate_T(a, c1, c2))

T = 10: S =  9, M =  9, S − M =  0 -> cặp  0, đơn  9, chi phí 90
T = 11: S = 12, M = 10, S − M =  2 -> cặp  2, đơn  8, chi phí 82
T = 12: S = 15, M = 11, S − M =  4 -> cặp  4, đơn  7, chi phí 74
T = 13: S = 18, M = 12, S − M =  6 -> cặp  6, đơn  6, chi phí 66
T = 14: S = 21, M = 13, S − M =  8 -> cặp  8, đơn  5, chi phí 58
T = 15: S = 24, M = 14, S − M = 10 -> cặp 10, đơn  4, chi phí 50
T = 16: S = 27, M = 15, S − M = 12 -> cặp 12, đơn  3, chi phí 42
T = 17: S = 30, M = 16, S − M = 14 -> cặp 14, đơn  2, chi phí 34
T = 18: S = 33, M = 17, S − M = 16 -> cặp 16, đơn  1, chi phí 26
T = 19: S = 36, M = 18, S − M = 18 -> cặp 18, đơn  0, chi phí 18
T = 20: S = 39, M = 19, S − M = 20 -> cặp 19, đơn  1, chi phí 29
T = 21: S = 42, M = 20, S − M = 22 -> cặp 21, đơn  0, chi phí 21
T = 22: S = 45, M = 21, S − M = 24 -> cặp 22, đơn  1, chi phí 32
đáp án: 18


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def minCostToEqualizeArray(self, nums: List[int], cost1: int, cost2: int) -> int:
        MOD = 10 ** 9 + 7
        n = len(nums)
        mx = max(nums)
        mn = min(nums)
        s = sum(nums)
        if n <= 2 or 2 * cost1 <= cost2:
            return (n * mx - s) * cost1 % MOD
        best = None
        for T in range(mx, 2 * mx + 1):
            S = n * T - s
            M = T - mn
            if M <= S - M:
                v = (S >> 1) * cost2 + (S & 1) * cost1
            else:
                v = (S - M) * cost2 + (2 * M - S) * cost1
            if best is None or v < best:
                best = v
        return best % MOD

for a, c1, c2, exp in examples:
    got = Solution().minCostToEqualizeArray(a, c1, c2)
    print(got, 'OK' if got == exp else 'SAI')

15 OK
6 OK
4 OK


## 10. Vì sao thuật toán đúng

1. **Đích:** chỉ tăng ⇒ `T ≥ max`; với `T` cố định, mọi lời giải là `p` thao tác đôi + `S − 2p` thao tác đơn, `p` khả thi ⇔ `p ≤ min(⌊S/2⌋, S − M)` (mục 3.1).
2. **Chọn `p`:** chi phí `p·cost2 + (S − 2p)·cost1 = S·cost1 + p·(cost2 − 2cost1)`. Nếu `cost2 < 2cost1` thì tăng `p` càng tốt ⇒ `p` lớn nhất; ngược lại `p = 0` và `T = max` tốt nhất (chi phí tỉ lệ `S`).
3. **Phạm vi `T`:** khi `T` tăng 1, `S` tăng `n` và `M` tăng 1 nên `(S − M) − M` tăng `n − 2 ≥ 1` (với `n ≥ 3`); tại `T = 2·max` ta có `S − M ≥ (n − 1)(2max) − (n−1)max − … ≥ M`, tức đã ở chế độ cân bằng. Trong chế độ cân bằng, chi phí là `⌊S/2⌋·cost2 + (S mod 2)·cost1` với `S` tăng `n` mỗi bước — qua hai bước tăng ít nhất `n·cost2 − cost1 > 0` ⇒ không cần xét thêm. Mục 12 đối chiếu với vét cạn và với phạm vi `[max, 5·max]`.
4. **Mod:** so sánh trên số nguyên thật, chỉ mod khi trả về.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn có nhớ | mũ | mũ |
| 2. Duyệt `T ∈ [max, 2max]` | `O(n + max)` | `O(1)` |
| 3. Điểm tới hạn | `O(n)` | `O(1)` |
| 4. Heap tham lam mỗi `T` | `O(max · n log n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'enumerate': enumerate_T, 'critical': critical_points, 'heap': greedy_heap,
         'submit': lambda a, c1, c2: Solution().minCostToEqualizeArray(a, c1, c2)}
edge = [([5], 3, 1, 0), ([2, 2, 2], 1, 1, 0), ([1, 10, 10], 10, 1, 18), ([1, 2, 3], 1, 5, 3), ([1, 1, 2], 5, 1, 1)]
for a, c1, c2, exp in examples + edge:
    if max(a) <= 10:
        assert brute(a, c1, c2, extra=10) == exp, (a, c1, c2, brute(a, c1, c2, extra=10))
    for name, f in impls.items():
        assert f(a, c1, c2) == exp, (name, a, c1, c2, f(a, c1, c2))
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `[1, 1, 2]`, `cost1 = 5`, `cost2 = 1`: với `T = 2` thiếu `[1, 1, 0]` ⇒ một thao tác đôi, chi phí `1`. `[1, 2, 3]` với `cost2 = 5 ≥ 2·cost1` ⇒ chỉ thao tác đơn: `2 + 1 = 3`. `[1, 10, 10]` ⇒ `18` (mục 4).

### 12.2. Đối chiếu ngẫu nhiên với vét cạn và với phạm vi `T` rộng hơn

In [10]:
cnt = 0
for _ in range(250):
    n = rng.randint(1, 4); a = [rng.randint(1, 4) for _ in range(n)]; c1 = rng.randint(1, 6); c2 = rng.randint(1, 12)
    exp = brute(a, c1, c2, extra=6)
    for name, f in impls.items():
        assert f(a, c1, c2) == exp, (name, a, c1, c2)
    cnt += 1
for _ in range(3000):
    n = rng.randint(1, 9); a = [rng.randint(1, 40) for _ in range(n)]; c1 = rng.randint(1, 30); c2 = rng.randint(1, 60)
    assert enumerate_T(a, c1, c2) == enumerate_T(a, c1, c2, upto=5) == critical_points(a, c1, c2)
print('Khớp vét cạn trên', cnt, 'bộ; phạm vi [max, 2max] = [max, 5max] = điểm tới hạn trên 3000 bộ.')

Khớp vét cạn trên 250 bộ; phạm vi [max, 2max] = [max, 5max] = điểm tới hạn trên 3000 bộ.


### 12.3. Ca lớn `n = 10^5`, giá trị và chi phí tới `10^6`

In [11]:
N = 10 ** 5
big = {
    'ngẫu nhiên': ([rng.randint(1, 10 ** 6) for _ in range(N)], rng.randint(1, 10 ** 6), rng.randint(1, 10 ** 6)),
    'một phần tử nhỏ': ([10 ** 6] * (N - 1) + [1], 10 ** 6, 1),
    'cost2 rẻ': ([rng.randint(1, 10 ** 6) for _ in range(N)], 10 ** 6, 1),
}
for name, (a, c1, c2) in big.items():
    t = time.time(); x = Solution().minCostToEqualizeArray(a, c1, c2); t1 = time.time() - t
    assert x == critical_points(a, c1, c2)
    print(f'{name:16s}: {x:10d} | nộp {t1:.2f}s')

ngẫu nhiên      :  586266191 | nộp 0.20s
một phần tử nhỏ :    2049999 | nộp 0.14s


cost2 rẻ        :  979785822 | nộp 0.15s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét đích `T = max(nums)`

In [12]:
def wrong_only_max(nums, c1, c2):
    n = len(nums); mx = max(nums)
    return cost_T(sum(nums), n, min(nums), c1, c2, mx) % MOD               # SAI

args = ([1, 10, 10], 10, 1)
print('sai :', wrong_only_max(*args), '| đúng:', Solution().minCostToEqualizeArray(*args))

sai : 90 | đúng: 18


**Vì sao sai:** nâng đích tạo thêm "đối tác" để ghép cặp với phần tử thiếu nhiều (mục 4).

### Lỗi 2 — Luôn ghép `⌊S/2⌋` cặp (bỏ qua giới hạn `S − M`)

In [13]:
def wrong_ignore_M(nums, c1, c2):
    n = len(nums); mx = max(nums); s = sum(nums)
    if 2 * c1 <= c2:
        return (n * mx - s) * c1 % MOD
    return min((n * T - s) // 2 * c2 + (n * T - s) % 2 * c1 for T in range(mx, 2 * mx + 1)) % MOD   # SAI

args = ([1, 10, 10], 10, 1)
print('sai :', wrong_ignore_M(*args), '| đúng:', Solution().minCostToEqualizeArray(*args))

sai : 6 | đúng: 18


**Vì sao sai:** với `T = 10`, thiếu `[9, 0, 0]` — **không thể** ghép cặp (cặp cần hai chỉ số khác nhau), nhưng công thức sai nhận 4 cặp giá 1.

### Lỗi 3 — Luôn ưu tiên thao tác đôi kể cả khi `cost2 ≥ 2·cost1`

In [14]:
def wrong_always_pairs(nums, c1, c2):
    n = len(nums); mx = max(nums); mn = min(nums); s = sum(nums)
    best = None
    for T in range(mx, 2 * mx + 1):
        S = n * T - s; M = T - mn; p = S // 2 if M <= S - M else S - M      # SAI: không xét 2c1 <= c2
        v = p * c2 + (S - 2 * p) * c1
        best = v if best is None else min(best, v)
    return best % MOD

args = ([1, 2, 3], 1, 5)
print('sai :', wrong_always_pairs(*args), '| đúng:', Solution().minCostToEqualizeArray(*args))

sai : 6 | đúng: 3


**Vì sao sai:** một thao tác đôi giá 5 thay cho hai thao tác đơn giá 1 + 1 — lỗ.

### Lỗi 4 — Lấy mod **trước** khi so sánh

In [15]:
def wrong_mod_before_min(nums, c1, c2):
    n = len(nums); mx = max(nums); mn = min(nums); s = sum(nums)
    if n <= 2 or 2 * c1 <= c2:
        return (n * mx - s) * c1 % MOD
    return min(cost_T(s, n, mn, c1, c2, T) % MOD for T in range(mx, 2 * mx + 1))   # SAI

found = None
for _ in range(2000):
    a = [rng.randint(1, 10 ** 6) for _ in range(rng.randint(3, 6))]; c1 = rng.randint(10 ** 5, 10 ** 6); c2 = rng.randint(1, 2 * c1 - 1)
    if wrong_mod_before_min(a, c1, c2) != Solution().minCostToEqualizeArray(a, c1, c2):
        found = (a, c1, c2); break
print('phản ví dụ:', found, '| sai :', wrong_mod_before_min(*found), '| đúng:', Solution().minCostToEqualizeArray(*found))

phản ví dụ: ([874060, 231915, 294175], 915047, 1536746) | sai : 159 | đúng: 106773794


**Vì sao sai:** chi phí thật có thể vượt `10^9 + 7`; sau khi mod, một chi phí **lớn** có thể trở thành số nhỏ ⇒ chọn sai. Phải tìm min trên số nguyên thật rồi mới mod.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chỉ được tăng, làm mọi phần tử bằng nhau" ⇒ duyệt giá trị đích `T ≥ max`.
- Thao tác tác động **hai phần tử khác nhau** ⇒ số cặp tối đa `min(⌊S/2⌋, S − max)` (bài "ghép cặp với ràng buộc phần tử lớn nhất" kinh điển).
- So sánh `cost2` với `2·cost1` trước tiên.
- Mod chỉ ở bước cuối khi có `min`.

## 15. Biến thể của bài

1. Thao tác tăng **ba** phần tử khác nhau (giá `cost3`) ⇒ số bộ ba tối đa `min(⌊S/3⌋, ⌊(S − M)/2⌋, …)` — phức tạp hơn.
2. Được cả **giảm** ⇒ đích là trung vị (nếu chỉ thao tác đơn).
3. Thao tác đôi trên hai phần tử **kề nhau** ⇒ trở thành bài khác (ràng buộc vị trí).
4. Chi phí thao tác đôi phụ thuộc cặp ⇒ bài luồng chi phí nhỏ nhất.

In [16]:
# Biến thể kiểm chứng: công thức số cặp tối đa với "ghép hai phần tử khác nhau" (mục 3.1) ở quy mô lớn hơn bằng tham lam
for _ in range(200):
    d = [rng.randint(0, 30) for _ in range(rng.randint(1, 7))]
    assert max_pairs_greedy(d) == min(sum(d) // 2, sum(d) - max(d))
print('min(⌊S/2⌋, S − M) khớp tham lam trên dữ liệu vừa.')

min(⌊S/2⌋, S − M) khớp tham lam trên dữ liệu vừa.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Vét cạn có nhớ | mũ | Oracle không giả thiết |
| 2 | Duyệt `T ∈ [max, 2max]` | `O(n + max)` | **Nộp bài** |
| 3 | Điểm tới hạn | `O(n)` | Nhanh nhất |
| 4 | Heap tham lam | `O(max · n log n)` | Kiểm chứng công thức |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[7, 7]`, `cost1 = 3`, `cost2 = 1`.

**Bài 2 (tính tay):** `[1, 3, 3]`, `cost1 = 4`, `cost2 = 3`.

**Bài 3 (điền TODO):** trả về luôn giá trị đích `T` nhỏ nhất đạt chi phí tối ưu.

**Bài 4 (tự viết lại):** số cặp tối đa `min(⌊S/2⌋, S − M)` — chứng minh và viết hàm dựng **danh sách cặp** cụ thể.

**Bài 5 (mini challenge):** phiên bản cho phép **giảm** một phần tử (giá `cost1`) và giảm hai phần tử (giá `cost2`) — chỉ giảm, không tăng.

In [17]:
def best_target_todo(nums, c1, c2):
    # TODO
    raise NotImplementedError

def build_pairs_todo(d):
    # TODO
    raise NotImplementedError

def decrease_version_todo(nums, c1, c2):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đã bằng nhau. (2) Không cần thao tác. (3) `0`.

**Bài 2.** (1) `T = 3`: thiếu `[2, 0, 0]` ⇒ `8`. (2) `T = 4`: thiếu `[3, 1, 1]`, `S = 5`, `M = 3 > 2` ⇒ 2 cặp + 1 đơn = `10`. `T = 5`: `[4, 2, 2]`, `S = 8`, `M = 4 ≤ 4` ⇒ 4 cặp = `12`. (3) `8`.

**Bài 3.** (1) Duyệt như lời giải, lưu `T` khi chi phí giảm **ngặt**. (2) Trả `T` đầu tiên đạt min. (3) Kiểm bằng duyệt.

**Bài 4.** (1) Lặp: lấy hai thiếu hụt lớn nhất, ghép một cặp. (2) Dùng heap. (3) Kiểm: số cặp = công thức, mỗi chỉ số dùng đúng `≤ d_i` lần.

**Bài 5.** (1) Đối xứng: giảm tới `T ≤ min`, thiếu hụt `nums[i] − T`. (2) `T ≥ 0`? đề không giới hạn — giả sử `T` bất kỳ `≤ min`. (3) Đổi dấu: `decrease(nums) = increase([−x for x in nums])`.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().minCostToEqualizeArray([7, 7], 3, 1) == 0
assert Solution().minCostToEqualizeArray([1, 3, 3], 4, 3) == 8 == brute([1, 3, 3], 4, 3, extra=6)

def best_target(nums, c1, c2):
    n = len(nums); mx = max(nums); mn = min(nums); s = sum(nums); best = None; bt = None
    for T in range(mx, 2 * mx + 1):
        v = (n * T - s) * c1 if (n <= 2 or 2 * c1 <= c2) else cost_T(s, n, mn, c1, c2, T)
        if best is None or v < best:
            best, bt = v, T
    return bt

assert best_target([1, 10, 10], 10, 1) == 19 and best_target([1, 3, 3], 4, 3) == 3
print('Bài 1: 0 | Bài 2: 8 | Bài 3: [1, 10, 10] -> T =', best_target([1, 10, 10], 10, 1))

Bài 1: 0 | Bài 2: 8 | Bài 3: [1, 10, 10] -> T = 19


In [19]:
# Bài 4, 5
def build_pairs(d):
    h = [(-v, i) for i, v in enumerate(d) if v > 0]; heapq.heapify(h); pairs = []
    while len(h) >= 2:
        a, i = heapq.heappop(h); b, j = heapq.heappop(h)
        pairs.append((i, j))
        if a + 1: heapq.heappush(h, (a + 1, i))
        if b + 1: heapq.heappush(h, (b + 1, j))
    return pairs

def decrease_version(nums, c1, c2):
    return Solution().minCostToEqualizeArray([max(nums) + 1 - x for x in nums], c1, c2)

def brute_decrease(nums, c1, c2, extra=6):
    mn = min(nums); best = None
    for T in range(mn, mn - extra - 1, -1):
        @lru_cache(maxsize=None)
        def f(d):
            if not any(d): return 0
            L = list(d); res = float('inf')
            for i in range(len(L)):
                if L[i]:
                    a = L[:]; a[i] -= 1; res = min(res, c1 + f(tuple(sorted(a))))
                    for j in range(i + 1, len(L)):
                        if L[j]:
                            b = L[:]; b[i] -= 1; b[j] -= 1; res = min(res, c2 + f(tuple(sorted(b))))
            return res
        v = f(tuple(sorted(x - T for x in nums))); best = v if best is None else min(best, v)
    return best % MOD

for _ in range(150):
    d = [rng.randint(0, 6) for _ in range(rng.randint(1, 6))]
    ps = build_pairs(d)
    assert len(ps) == min(sum(d) // 2, sum(d) - max(d)) and all(i != j for i, j in ps)
    use = [0] * len(d)
    for i, j in ps:
        use[i] += 1; use[j] += 1
    assert all(u <= v for u, v in zip(use, d))
for _ in range(120):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 4))]; c1 = rng.randint(1, 6); c2 = rng.randint(1, 12)
    assert decrease_version(a, c1, c2) == brute_decrease(a, c1, c2)
print('Bài 4 (dựng cặp) và Bài 5 (phiên bản giảm = đổi dấu) khớp brute.')

Bài 4 (dựng cặp) và Bài 5 (phiên bản giảm = đổi dấu) khớp brute.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
if n <= 2 or 2*c1 <= c2: return (n*mx − s)*c1 % MOD
for T in range(mx, 2*mx + 1):
    S = n*T − s; M = T − mn
    v = (S//2)*c2 + (S%2)*c1 if M <= S − M else (S − M)*c2 + (2M − S)*c1
return min(v) % MOD
```

**Checklist:**

- [ ] So `2·cost1` với `cost2` trước?
- [ ] Giới hạn số cặp bởi `S − M`?
- [ ] Thử `T > max`?
- [ ] Mod **sau** khi lấy min?